# Data Merge
>We will merge and treat all the preprocessed dataframes to det the final dataset

In [37]:
import pandas as pd
import numpy as np

In [38]:
semeval_e_train = pd.read_csv('../data/raw/English/E-c/2018-E-c-En-train.txt', sep='\t')
semeval_e_test = pd.read_csv('../data/raw/English/E-c/2018-E-c-En-test-gold.txt', sep='\t')
semeval_e_dev = pd.read_csv('../data/raw/English/E-c/2018-E-c-En-dev.txt', sep='\t')
semeval_e = pd.concat([semeval_e_train, semeval_e_test, semeval_e_dev])
semeval_e.drop(['ID'], axis=1, inplace=True)
emotion_cols = semeval_e.columns.drop('Tweet')
for col in emotion_cols:
    semeval_e[col] = semeval_e[col].replace({1: col, 0: ""})
semeval_e["emociones_merged"] = semeval_e[emotion_cols].apply(
    lambda fila: [val for val in fila if val != ""][0] if sum(fila != "") == 1 
                 else [val for val in fila if val != ""],
    axis=1
)
semeval_e.drop(emotion_cols, axis=1, inplace=True)
semeval_e.rename(columns={
    'Tweet': 'text',
    'emociones_merged': 'emotion'
}, inplace=True)
emotion_cols

Index(['anger', 'anticipation', 'disgust', 'fear', 'joy', 'love', 'optimism',
       'pessimism', 'sadness', 'surprise', 'trust'],
      dtype='object')

In [39]:
crowdflower = pd.read_csv('../data/preprocessed/crowdflower.csv', usecols=[1, 2], names=['text', 'emotion']).drop(0)
go_emotions = pd.read_csv('../data/preprocessed/goemotions_full_clean_final.csv', names=['text', 'emotion']).drop(0)
isear = pd.read_csv('../data/preprocessed/isear.csv', usecols=[1, 2], names=['text', 'emotion']).drop(0)
kushagra = pd.read_csv('../data/preprocessed/kushagra.csv', usecols=[1, 2], names=['text', 'emotion']).drop(0)
meld = pd.read_csv('../data/preprocessed/meld.csv', usecols=[1, 2], names=['text', 'emotion']).drop(0)
meld_dya = pd.read_csv('../data/preprocessed/meld-dya.csv', usecols=[1, 2], names=['text', 'emotion']).drop(0)
meld_emorynlp = pd.read_csv('../data/preprocessed/meld-emorynlp.csv', usecols=[1, 2], names=['text', 'emotion']).drop(0)
reddit = pd.read_csv('../data/preprocessed/redditposts.csv', names=['text', 'emotion']).drop(0)
semeval_ei = pd.read_csv('../data/preprocessed/semeval-ei.csv', usecols=[1, 2], names=['text', 'emotion']).drop(0)
semeval_e_final = pd.read_csv('../data/preprocessed/semeval-e-final.csv', names=['text', 'emotion']).drop(0)
twitter = pd.read_csv('../data/preprocessed/twitter-emotion.csv', usecols=[1, 2], names=['text', 'emotion']).drop(0)



In [40]:
full_data = pd.concat([crowdflower, go_emotions, isear, kushagra, 
                       meld, meld_dya, meld_emorynlp, reddit, semeval_ei,
                       semeval_e_final, twitter])

In [41]:
full_data = full_data.drop_duplicates(subset=['text'])
full_data


,text,emotion
1,@tiffanylue i know i was listenin to bad habi...,empty
2,Layin n bed with a headache ughhhh...waitin o...,sadness
3,Funeral ceremony...gloomy friday...,sadness
4,wants to hang out with friends SOON!,enthusiasm
5,@dannycastillo We want to trade with someone w...,neutral
...,...,...
6073,@kamaalrkhan Which #chutiya #producer #investe...,anger
6074,Russia story will infuriate Trump today. Media...,anger
6075,"@AleOfATime It would be shocking, but it's sim...",anger
6079,@BadHombreNPS @SecretaryPerry If this didn't m...,anger


In [42]:
full_data['emotion'].unique()

array(['empty', 'sadness', 'enthusiasm', 'neutral', 'worry', 'surprise',
       'love', 'fun', 'hate', 'happiness', 'boredom', 'relief', 'anger',
       'sad', 'joy', 'fear', 'suprise', 'disgust', 'Joyful', 'Neutral',
       'Powerful', 'Mad', 'Sad', 'Scared', 'Peaceful', 'happy', 'angry',
       'rant', 'afraid', 'funny', 'surprised'], dtype=object)

In [43]:
full_data.replace(['empty', 'sadness', 'Sad'], 'sad', inplace=True)
full_data.replace(['enthusiasm', 'fun', 'happiness', 'Joyful', 'happy', 'funny', 'love'],
                  'joy', inplace=True)
full_data.replace(['worry', 'Scared', 'afraid'], 'fear', inplace=True)
full_data.replace(['suprise', 'surprised'], 'surprise', inplace=True)
full_data.replace(['hate', 'disgust', 'Mad', 'rant', 'angry'], 'anger', inplace=True)
full_data.replace(['neutral', 'boredom', 'relief', 'Neutral', 'Powerful', 'Peaceful'], 
                  np.nan, inplace=True)
full_data.dropna(inplace=True)


In [44]:
full_data['emotion'].unique()

array(['sad', 'joy', 'fear', 'surprise', 'anger'], dtype=object)

In [45]:
full_data['emotion'].value_counts()

emotion
joy         208380
sad         132290
anger        76359
fear         60154
surprise     19039
Name: count, dtype: int64

In [46]:
full_data

,text,emotion
1,@tiffanylue i know i was listenin to bad habi...,sad
2,Layin n bed with a headache ughhhh...waitin o...,sad
3,Funeral ceremony...gloomy friday...,sad
4,wants to hang out with friends SOON!,joy
6,Re-pinging @ghostridah14: why didn't you go to...,fear
...,...,...
6073,@kamaalrkhan Which #chutiya #producer #investe...,anger
6074,Russia story will infuriate Trump today. Media...,anger
6075,"@AleOfATime It would be shocking, but it's sim...",anger
6079,@BadHombreNPS @SecretaryPerry If this didn't m...,anger


In [48]:
full_data.to_csv("../data/processed/full_data.csv", index=False)